In [0]:
# Load permanent CBSL Gold source tables

cbsl_telecom_df = spark.table(
    "workspace.census360.gold_cbsl_telecom_analytical_v1"
)

cbsl_hies_df = spark.table(
    "workspace.census360.gold_cbsl_hies_analytical_v1"
)

cbsl_national_poverty_df = spark.table(
    "workspace.census360.gold_cbsl_poverty_national_analytical_v1"
)

cbsl_province_poverty_df = spark.table(
    "workspace.census360.gold_cbsl_poverty_province_analytical_v1"
)

cbsl_province_socioeconomic_df = spark.table(
    "workspace.census360.gold_cbsl_provincial_socioeconomic_analytical_v1"
)

cbsl_province_joined_df = spark.table(
    "workspace.census360.gold_cbsl_province_poverty_socioeconomic_joined_v1"
)

cbsl_province_association_df = spark.table(
    "workspace.census360.gold_cbsl_province_association_summary_v1"
)

cbsl_national_overlap_df = spark.table(
    "workspace.census360.gold_cbsl_national_overlap_summary_v1"
)

print("Successfully loaded all 8 CBSL Gold dashboard source tables.")

In [0]:
source_tables = {
    "Telecom": "workspace.census360.gold_cbsl_telecom_analytical_v1",
    "HIES": "workspace.census360.gold_cbsl_hies_analytical_v1",
    "National Poverty": "workspace.census360.gold_cbsl_poverty_national_analytical_v1",
    "Provincial Poverty": "workspace.census360.gold_cbsl_poverty_province_analytical_v1",
    "Provincial Socioeconomic": "workspace.census360.gold_cbsl_provincial_socioeconomic_analytical_v1",
    "Province Joined": "workspace.census360.gold_cbsl_province_poverty_socioeconomic_joined_v1",
    "Association Summary": "workspace.census360.gold_cbsl_province_association_summary_v1",
    "National Overlap Summary": "workspace.census360.gold_cbsl_national_overlap_summary_v1"
}

for name, table_name in source_tables.items():
    exists = spark.catalog.tableExists(table_name)

    if exists:
        df = spark.table(table_name)
        print(f"{name}: Exists=True | Rows={df.count()} | Columns={len(df.columns)}")
    else:
        print(f"{name}: Exists=False")

In [0]:
print("Telecom Gold table columns:\n")

for column_name in cbsl_telecom_df.columns:
    print(column_name)

In [0]:
from pyspark.sql import functions as F

cbsl_dashboard_telecom_trend_df = (
    cbsl_telecom_df
    .select(
        F.col("year").alias("analysis_year"),
        F.col("data_status").alias("observation_status"),
        F.col("is_revised_observation"),
        F.col("is_provisional_observation"),

        F.col("internet_connections_total")
         .alias("internet_connections_including_mobile_total"),

        F.col("internet_connections_yoy_change"),
        F.col("internet_connections_yoy_growth_percent"),
        F.col("internet_growth_3yr_trailing_average_percent"),

        F.col("cellular_phones_thousand"),
        F.col("cellular_penetration_per_100"),
        F.col("cellular_penetration_change_pp"),

        F.col("fixed_subscriber_base_thousand")
         .alias("fixed_access_subscribers_thousand"),

        F.col("fixed_subscriber_growth_percent")
         .alias("fixed_access_growth_percent"),

        F.col("telephone_penetration_per_100"),
        F.col("wireline_telephones_thousand"),
        F.col("wireless_local_loop_thousand"),

        F.col("internet_metric_scope"),
        F.col("fixed_metric_scope"),

        F.when(
            F.col("is_provisional_observation") == True,
            F.lit("Provisional data")
        ).when(
            F.col("is_revised_observation") == True,
            F.lit("Revised data")
        ).otherwise(
            F.lit("Published data")
        ).alias("dashboard_status_label")
    )
    .orderBy("analysis_year")
)

print(
    f"Telecom dashboard dataset created: "
    f"{cbsl_dashboard_telecom_trend_df.count()} rows, "
    f"{len(cbsl_dashboard_telecom_trend_df.columns)} columns"
)

display(cbsl_dashboard_telecom_trend_df)

In [0]:
from pyspark.sql import functions as F

telecom_validation = (
    cbsl_dashboard_telecom_trend_df
    .agg(
        F.count("*").alias("row_count"),
        F.min("analysis_year").alias("minimum_year"),
        F.max("analysis_year").alias("maximum_year"),
        F.countDistinct("analysis_year").alias("distinct_years"),
        F.sum(
            F.when(F.col("internet_connections_including_mobile_total").isNull(), 1).otherwise(0)
        ).alias("null_internet_connections"),
        F.sum(
            F.when(F.col("cellular_penetration_per_100").isNull(), 1).otherwise(0)
        ).alias("null_cellular_penetration"),
        F.sum(
            F.when(F.col("fixed_access_subscribers_thousand").isNull(), 1).otherwise(0)
        ).alias("null_fixed_access_subscribers")
    )
)

duplicate_years = (
    cbsl_dashboard_telecom_trend_df
    .groupBy("analysis_year")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(telecom_validation)
print(f"Duplicate year groups: {duplicate_years}")

In [0]:
telecom_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_telecom_trend_v1"
)

if spark.catalog.tableExists(telecom_dashboard_target):
    print(f"Table already exists: {telecom_dashboard_target}")
else:
    print(f"Table name is available: {telecom_dashboard_target}")

In [0]:
print("Columns:")
for column_name in cbsl_dashboard_telecom_trend_df.columns:
    print(column_name)

print("\nRepresentative rows:")
display(
    cbsl_dashboard_telecom_trend_df
    .orderBy("analysis_year")
    .limit(5)
)

In [0]:
telecom_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_telecom_trend_v1"
)

if spark.catalog.tableExists(telecom_dashboard_target):
    print(f"Table already exists. Nothing was written: {telecom_dashboard_target}")
else:
    (
        cbsl_dashboard_telecom_trend_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(telecom_dashboard_target)
    )

    print(f"Table created successfully: {telecom_dashboard_target}")

In [0]:
telecom_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_telecom_trend_v1"
)

saved_telecom_dashboard_df = spark.table(telecom_dashboard_target)

print(f"Table: {telecom_dashboard_target}")
print(f"Exists: {spark.catalog.tableExists(telecom_dashboard_target)}")
print(f"Rows: {saved_telecom_dashboard_df.count()}")
print(f"Columns: {len(saved_telecom_dashboard_df.columns)}")

In [0]:
print("HIES Gold table columns:\n")

for column_name in cbsl_hies_df.columns:
    print(column_name)

In [0]:
from pyspark.sql import functions as F

cbsl_dashboard_hies_trend_df = (
    cbsl_hies_df
    .select(
        F.col("survey_period"),
        F.col("representative_year").alias("analysis_year"),
        F.col("period_start_year"),
        F.col("period_end_year"),
        F.col("survey_period_type"),
        F.col("years_since_previous_survey"),
        F.col("is_original_single_year_observation"),
        F.col("is_multi_year_survey_observation"),
        F.col("is_derived_annual_value"),
        F.col("overlaps_cbsl_telecom_period"),

        F.col("mean_household_income_monthly_lkr")
         .alias("mean_household_income_lkr"),

        F.col("median_household_income_monthly_lkr")
         .alias("median_household_income_lkr"),

        F.col("mean_per_capita_income_monthly_lkr")
         .alias("mean_income_per_person_lkr"),

        F.col("mean_household_expenditure_monthly_lkr")
         .alias("mean_household_expenditure_lkr"),

        F.col("food_drink_expenditure_monthly_lkr")
         .alias("food_and_drink_expenditure_lkr"),

        F.col("non_food_expenditure_monthly_lkr")
         .alias("non_food_expenditure_lkr"),

        F.col("food_ratio_percent"),
        F.col("gini_household_income")
         .alias("household_income_gini"),

        F.col("household_size"),
        F.col("income_receivers_per_household"),

        F.col("mean_household_income_change_percent")
         .alias("household_income_change_percent"),

        F.col("mean_household_expenditure_change_percent")
         .alias("household_expenditure_change_percent"),

        F.col("mean_household_income_annualized_growth_percent")
         .alias("annualized_household_income_growth_percent"),

        F.col("mean_household_expenditure_annualized_growth_percent")
         .alias("annualized_household_expenditure_growth_percent"),

        F.lit("Nominal values; not adjusted for inflation")
         .alias("value_interpretation_note"),

        F.lit("Irregular survey observations; do not interpret as annual data")
         .alias("time_series_interpretation_note")
    )
    .orderBy("analysis_year")
)

print(
    f"HIES dashboard dataset created: "
    f"{cbsl_dashboard_hies_trend_df.count()} rows, "
    f"{len(cbsl_dashboard_hies_trend_df.columns)} columns"
)

display(cbsl_dashboard_hies_trend_df)

In [0]:
from pyspark.sql import functions as F

hies_validation = (
    cbsl_dashboard_hies_trend_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("survey_period").alias("distinct_survey_periods"),
        F.min("analysis_year").alias("minimum_representative_year"),
        F.max("analysis_year").alias("maximum_representative_year"),

        F.sum(
            F.when(F.col("is_derived_annual_value") == True, 1).otherwise(0)
        ).alias("invented_annual_value_rows"),

        F.sum(
            F.when(F.col("mean_household_income_lkr").isNull(), 1).otherwise(0)
        ).alias("null_mean_household_income"),

        F.sum(
            F.when(F.col("mean_household_expenditure_lkr").isNull(), 1).otherwise(0)
        ).alias("null_household_expenditure"),

        F.sum(
            F.when(F.col("mean_income_per_person_lkr").isNull(), 1).otherwise(0)
        ).alias("null_income_per_person")
    )
)

duplicate_periods = (
    cbsl_dashboard_hies_trend_df
    .groupBy("survey_period")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(hies_validation)
print(f"Duplicate survey-period groups: {duplicate_periods}")

In [0]:
hies_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_hies_trend_v1"
)

if spark.catalog.tableExists(hies_dashboard_target):
    print(f"Table already exists: {hies_dashboard_target}")
else:
    print(f"Table name is available: {hies_dashboard_target}")

In [0]:
print("Columns:")
for column_name in cbsl_dashboard_hies_trend_df.columns:
    print(column_name)

print("\nRepresentative rows:")
display(
    cbsl_dashboard_hies_trend_df
    .orderBy("analysis_year")
    .limit(5)
)

In [0]:
hies_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_hies_trend_v1"
)

if spark.catalog.tableExists(hies_dashboard_target):
    print(f"Table already exists. Nothing was written: {hies_dashboard_target}")
else:
    (
        cbsl_dashboard_hies_trend_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(hies_dashboard_target)
    )

    print(f"Table created successfully: {hies_dashboard_target}")

In [0]:
hies_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_hies_trend_v1"
)

saved_hies_dashboard_df = spark.table(hies_dashboard_target)

print(f"Table: {hies_dashboard_target}")
print(f"Exists: {spark.catalog.tableExists(hies_dashboard_target)}")
print(f"Rows: {saved_hies_dashboard_df.count()}")
print(f"Columns: {len(saved_hies_dashboard_df.columns)}")

In [0]:
from pyspark.sql import functions as F

telecom_insight_source_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_telecom_trend_v1"
)

first_observation = (
    telecom_insight_source_df
    .orderBy(F.col("analysis_year").asc())
    .first()
)

latest_observation = (
    telecom_insight_source_df
    .orderBy(F.col("analysis_year").desc())
    .first()
)

peak_internet_growth = (
    telecom_insight_source_df
    .filter(F.col("internet_connections_yoy_growth_percent").isNotNull())
    .orderBy(F.col("internet_connections_yoy_growth_percent").desc())
    .first()
)

largest_cellular_penetration_gain = (
    telecom_insight_source_df
    .filter(F.col("cellular_penetration_change_pp").isNotNull())
    .orderBy(F.col("cellular_penetration_change_pp").desc())
    .first()
)

internet_period_growth_percent = (
    (
        latest_observation["internet_connections_including_mobile_total"]
        / first_observation["internet_connections_including_mobile_total"]
    ) - 1
) * 100

fixed_access_period_growth_percent = (
    (
        latest_observation["fixed_access_subscribers_thousand"]
        / first_observation["fixed_access_subscribers_thousand"]
    ) - 1
) * 100

cellular_penetration_period_change = (
    latest_observation["cellular_penetration_per_100"]
    - first_observation["cellular_penetration_per_100"]
)

fixed_access_direction = (
    "increased"
    if fixed_access_period_growth_percent >= 0
    else "decreased"
)

telecom_insight_rows = [
    (
        "Long-term internet connection growth",
        f"{internet_period_growth_percent:,.1f}%",
        f"{first_observation['analysis_year']}–{latest_observation['analysis_year']}",
        "Internet connections expanded substantially across the period. "
        "This measure includes mobile internet connections."
    ),
    (
        "Highest annual internet growth",
        f"{peak_internet_growth['internet_connections_yoy_growth_percent']:,.1f}%",
        str(peak_internet_growth["analysis_year"]),
        "This was the strongest observed year-over-year increase in the available series."
    ),
    (
        "Cellular penetration change",
        f"{cellular_penetration_period_change:+,.1f} percentage points",
        f"{first_observation['analysis_year']}–{latest_observation['analysis_year']}",
        "Shows the long-term change in cellular subscriptions per 100 people."
    ),
    (
        "Fixed-access subscriber movement",
        f"{fixed_access_period_growth_percent:+,.1f}%",
        f"{first_observation['analysis_year']}–{latest_observation['analysis_year']}",
        f"The fixed-access subscriber indicator {fixed_access_direction} over the period."
    ),
    (
        "Latest telecom observation",
        str(latest_observation["analysis_year"]),
        latest_observation["dashboard_status_label"],
        "The latest values must be interpreted together with their publication status."
    )
]

cbsl_dashboard_telecom_insights_df = spark.createDataFrame(
    telecom_insight_rows,
    [
        "insight_title",
        "insight_value",
        "analysis_period",
        "client_interpretation"
    ]
)

display(cbsl_dashboard_telecom_insights_df)

In [0]:
from pyspark.sql import functions as F

hies_insight_source_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_hies_trend_v1"
)

first_hies = (
    hies_insight_source_df
    .orderBy(F.col("analysis_year").asc())
    .first()
)

latest_hies = (
    hies_insight_source_df
    .orderBy(F.col("analysis_year").desc())
    .first()
)

highest_income_growth = (
    hies_insight_source_df
    .filter(F.col("annualized_household_income_growth_percent").isNotNull())
    .orderBy(F.col("annualized_household_income_growth_percent").desc())
    .first()
)

income_growth_percent = (
    (
        latest_hies["mean_household_income_lkr"]
        / first_hies["mean_household_income_lkr"]
    ) - 1
) * 100

expenditure_growth_percent = (
    (
        latest_hies["mean_household_expenditure_lkr"]
        / first_hies["mean_household_expenditure_lkr"]
    ) - 1
) * 100

latest_income_expenditure_gap = (
    latest_hies["mean_household_income_lkr"]
    - latest_hies["mean_household_expenditure_lkr"]
)

hies_insight_rows = [
    (
        "Long-term household income growth",
        f"{income_growth_percent:,.1f}%",
        f"{first_hies['survey_period']}–{latest_hies['survey_period']}",
        "Mean monthly household income increased substantially, but the values are nominal and not adjusted for inflation."
    ),
    (
        "Long-term household expenditure growth",
        f"{expenditure_growth_percent:,.1f}%",
        f"{first_hies['survey_period']}–{latest_hies['survey_period']}",
        "Household expenditure also increased substantially in nominal terms."
    ),
    (
        "Latest income-expenditure gap",
        f"LKR {latest_income_expenditure_gap:,.0f}",
        latest_hies["survey_period"],
        "Difference between latest mean monthly household income and expenditure; it is not a direct measure of household savings."
    ),
    (
        "Latest food expenditure share",
        f"{latest_hies['food_ratio_percent']:,.1f}%",
        latest_hies["survey_period"],
        "Shows the percentage of household expenditure allocated to food and drink."
    ),
    (
        "Highest annualized income growth",
        f"{highest_income_growth['annualized_household_income_growth_percent']:,.1f}%",
        highest_income_growth["survey_period"],
        "Strongest annualized nominal household-income growth between consecutive surveys."
    ),
    (
        "Latest household income inequality",
        f"{latest_hies['household_income_gini']:,.3f}",
        latest_hies["survey_period"],
        "Higher Gini values indicate greater income inequality."
    )
]

cbsl_dashboard_hies_insights_df = spark.createDataFrame(
    hies_insight_rows,
    [
        "insight_title",
        "insight_value",
        "analysis_period",
        "client_interpretation"
    ]
)

display(cbsl_dashboard_hies_insights_df)

In [0]:
print("National poverty Gold table columns:\n")

for column_name in cbsl_national_poverty_df.columns:
    print(column_name)

In [0]:
from pyspark.sql import functions as F

cbsl_dashboard_national_poverty_df = (
    cbsl_national_poverty_df
    .select(
        F.col("survey_period"),
        F.col("representative_year").alias("analysis_year"),
        F.col("definition_variant"),
        F.col("poverty_definition_group"),
        F.col("comparison_series"),
        F.col("safe_for_historical_change_analysis"),
        F.col("requires_definition_warning"),

        F.col("poverty_headcount_index_percent")
         .alias("poverty_headcount_percent"),

        F.col("poor_household_percentage")
         .alias("poor_household_percent"),

        F.col("poverty_gap_index_percent")
         .alias("poverty_gap_percent"),

        F.col("poverty_headcount_change_pp"),

        F.when(
            F.col("safe_for_historical_change_analysis") == True,
            F.lit("Comparable historical series")
        ).otherwise(
            F.lit("Separate definition observation")
        ).alias("comparison_status"),

        F.when(
            F.col("requires_definition_warning") == True,
            F.lit(
                "Uses a different poverty-line definition; "
                "do not connect directly to the older comparable series."
            )
        ).otherwise(
            F.lit(
                "Comparable with 2012/13, 2016 and 2019(a) "
                "under the older poverty-line series."
            )
        ).alias("definition_warning"),

        F.concat(
            F.col("survey_period"),
            F.lit(" — "),
            F.col("definition_variant")
        ).alias("dashboard_observation_label")
    )
    .orderBy("analysis_year", "definition_variant")
)

print(
    f"National poverty dashboard dataset created: "
    f"{cbsl_dashboard_national_poverty_df.count()} rows, "
    f"{len(cbsl_dashboard_national_poverty_df.columns)} columns"
)

display(cbsl_dashboard_national_poverty_df)

In [0]:
from pyspark.sql import functions as F

poverty_validation = (
    cbsl_dashboard_national_poverty_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("dashboard_observation_label")
         .alias("distinct_observations"),

        F.sum(
            F.when(
                F.col("safe_for_historical_change_analysis") == True, 1
            ).otherwise(0)
        ).alias("comparable_series_rows"),

        F.sum(
            F.when(
                F.col("requires_definition_warning") == True, 1
            ).otherwise(0)
        ).alias("separate_definition_rows"),

        F.sum(
            F.when(
                F.col("poverty_headcount_percent").isNull(), 1
            ).otherwise(0)
        ).alias("null_poverty_headcount_values")
    )
)

duplicate_observations = (
    cbsl_dashboard_national_poverty_df
    .groupBy("dashboard_observation_label")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(poverty_validation)
print(f"Duplicate observation groups: {duplicate_observations}")

In [0]:
national_poverty_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_national_poverty_v1"
)

if spark.catalog.tableExists(national_poverty_dashboard_target):
    print(f"Table already exists: {national_poverty_dashboard_target}")
else:
    print(f"Table name is available: {national_poverty_dashboard_target}")

In [0]:
print("Columns:")
for column_name in cbsl_dashboard_national_poverty_df.columns:
    print(column_name)

print("\nAll national poverty observations:")
display(
    cbsl_dashboard_national_poverty_df
    .orderBy("analysis_year", "definition_variant")
)

In [0]:
national_poverty_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_national_poverty_v1"
)

if spark.catalog.tableExists(national_poverty_dashboard_target):
    print(
        f"Table already exists. Nothing was written: "
        f"{national_poverty_dashboard_target}"
    )
else:
    (
        cbsl_dashboard_national_poverty_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(national_poverty_dashboard_target)
    )

    print(
        f"Table created successfully: "
        f"{national_poverty_dashboard_target}"
    )

In [0]:
national_poverty_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_national_poverty_v1"
)

saved_national_poverty_df = spark.table(
    national_poverty_dashboard_target
)

print(f"Table: {national_poverty_dashboard_target}")
print(
    f"Exists: "
    f"{spark.catalog.tableExists(national_poverty_dashboard_target)}"
)
print(f"Rows: {saved_national_poverty_df.count()}")
print(f"Columns: {len(saved_national_poverty_df.columns)}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

province_source_df = (
    cbsl_province_joined_df
    .select(
        F.col("geography_name").alias("province_name"),
        F.col("analysis_year"),
        F.col("survey_period"),

        F.col("poverty_headcount_index_percent")
         .alias("poverty_headcount_percent"),

        F.col("poverty_gap_index_percent")
         .alias("poverty_gap_percent"),

        F.col("mean_income_per_household_monthly_lkr")
         .alias("mean_household_income_lkr"),

        F.col("mean_income_per_person_monthly_lkr")
         .alias("mean_income_per_person_lkr"),

        F.col("median_income_per_household_monthly_lkr")
         .alias("median_household_income_lkr"),

        F.col("household_expenditure_monthly_lkr")
         .alias("household_expenditure_lkr"),

        F.col("expenditure_share_communication_percent")
         .alias("communication_expenditure_share_percent"),

        F.col("education_passed_gce_ol_percent")
         .alias("gce_ol_attainment_percent"),

        F.col("education_passed_gce_al_and_above_percent")
         .alias("gce_al_and_above_attainment_percent"),

        F.col("gini_coefficient_households")
         .alias("household_gini"),

        F.col("poverty_headcount_change_pp")
    )
)

province_benchmarks_df = (
    province_source_df
    .groupBy("analysis_year")
    .agg(
        F.expr(
            "percentile_approx(poverty_headcount_percent, 0.5)"
        ).alias("median_provincial_poverty_percent"),

        F.expr(
            "percentile_approx(mean_income_per_person_lkr, 0.5)"
        ).alias("median_provincial_income_per_person_lkr"),

        F.expr(
            "percentile_approx(household_expenditure_lkr, 0.5)"
        ).alias("median_provincial_expenditure_lkr")
    )
)

poverty_rank_window = Window.partitionBy("analysis_year").orderBy(
    F.col("poverty_headcount_percent").desc()
)

income_rank_window = Window.partitionBy("analysis_year").orderBy(
    F.col("mean_income_per_person_lkr").desc()
)

cbsl_dashboard_province_comparison_df = (
    province_source_df
    .join(
        province_benchmarks_df,
        on="analysis_year",
        how="left"
    )
    .withColumn(
        "poverty_rank",
        F.dense_rank().over(poverty_rank_window)
    )
    .withColumn(
        "income_rank",
        F.dense_rank().over(income_rank_window)
    )
    .withColumn(
        "income_poverty_rank_gap",
        F.abs(F.col("income_rank") - F.col("poverty_rank"))
    )
    .withColumn(
        "poverty_vs_median_pp",
        F.round(
            F.col("poverty_headcount_percent")
            - F.col("median_provincial_poverty_percent"),
            2
        )
    )
    .withColumn(
        "income_per_person_vs_median_lkr",
        F.round(
            F.col("mean_income_per_person_lkr")
            - F.col("median_provincial_income_per_person_lkr"),
            0
        )
    )
    .withColumn(
        "province_segment",
        F.when(
            (F.col("poverty_headcount_percent")
             > F.col("median_provincial_poverty_percent"))
            &
            (F.col("mean_income_per_person_lkr")
             < F.col("median_provincial_income_per_person_lkr")),
            F.lit("Higher poverty / Lower income")
        ).when(
            (F.col("poverty_headcount_percent")
             <= F.col("median_provincial_poverty_percent"))
            &
            (F.col("mean_income_per_person_lkr")
             >= F.col("median_provincial_income_per_person_lkr")),
            F.lit("Lower poverty / Higher income")
        ).otherwise(
            F.lit("Mixed socioeconomic position")
        )
    )
    .orderBy("analysis_year", "poverty_rank")
)

print(
    f"Province comparison dataset created: "
    f"{cbsl_dashboard_province_comparison_df.count()} rows, "
    f"{len(cbsl_dashboard_province_comparison_df.columns)} columns"
)

display(cbsl_dashboard_province_comparison_df)

In [0]:
from pyspark.sql import functions as F

province_validation = (
    cbsl_dashboard_province_comparison_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("analysis_year").alias("distinct_years"),
        F.countDistinct("province_name").alias("distinct_provinces"),

        F.sum(
            F.when(F.col("poverty_headcount_percent").isNull(), 1).otherwise(0)
        ).alias("null_poverty_values"),

        F.sum(
            F.when(F.col("mean_income_per_person_lkr").isNull(), 1).otherwise(0)
        ).alias("null_income_values"),

        F.sum(
            F.when(F.col("province_segment").isNull(), 1).otherwise(0)
        ).alias("null_segments")
    )
)

duplicate_province_years = (
    cbsl_dashboard_province_comparison_df
    .groupBy("analysis_year", "province_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

rank_validation = (
    cbsl_dashboard_province_comparison_df
    .groupBy("analysis_year")
    .agg(
        F.countDistinct("province_name").alias("province_count"),
        F.min("poverty_rank").alias("minimum_poverty_rank"),
        F.max("poverty_rank").alias("maximum_poverty_rank"),
        F.min("income_rank").alias("minimum_income_rank"),
        F.max("income_rank").alias("maximum_income_rank")
    )
    .orderBy("analysis_year")
)

display(province_validation)
print(f"Duplicate province-year groups: {duplicate_province_years}")
display(rank_validation)

In [0]:
from pyspark.sql import functions as F

duplicate_province_years = (
    cbsl_dashboard_province_comparison_df
    .groupBy("analysis_year", "province_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate province-year groups: {duplicate_province_years}")

display(
    cbsl_dashboard_province_comparison_df
    .groupBy("analysis_year", "province_segment")
    .agg(
        F.count("*").alias("province_count")
    )
    .orderBy("analysis_year", "province_segment")
)

In [0]:
display(
    cbsl_dashboard_province_comparison_df
    .filter(F.col("analysis_year") == 2019)
    .select(
        "province_name",
        "poverty_headcount_percent",
        "mean_income_per_person_lkr",
        "median_provincial_poverty_percent",
        "median_provincial_income_per_person_lkr",
        "poverty_rank",
        "income_rank",
        "income_poverty_rank_gap",
        "province_segment"
    )
    .orderBy("poverty_rank")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

poverty_outcome_rank_window = (
    Window
    .partitionBy("analysis_year")
    .orderBy(F.col("poverty_headcount_percent").asc())
)

cbsl_dashboard_province_comparison_df = (
    cbsl_dashboard_province_comparison_df
    .drop("income_poverty_rank_gap")
    .withColumn(
        "poverty_outcome_rank",
        F.dense_rank().over(poverty_outcome_rank_window)
    )
    .withColumn(
        "income_poverty_rank_gap",
        F.abs(
            F.col("income_rank") - F.col("poverty_outcome_rank")
        )
    )
)

display(
    cbsl_dashboard_province_comparison_df
    .filter(F.col("analysis_year") == 2019)
    .select(
        "province_name",
        "poverty_headcount_percent",
        "mean_income_per_person_lkr",
        "poverty_rank",
        "poverty_outcome_rank",
        "income_rank",
        "income_poverty_rank_gap",
        "province_segment"
    )
    .orderBy("poverty_rank")
)

In [0]:
province_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_province_comparison_v1"
)

if spark.catalog.tableExists(province_dashboard_target):
    print(f"Table already exists: {province_dashboard_target}")
else:
    print(f"Table name is available: {province_dashboard_target}")

In [0]:
print("Columns:")
for column_name in cbsl_dashboard_province_comparison_df.columns:
    print(column_name)

print("\n2019 provincial comparison:")
display(
    cbsl_dashboard_province_comparison_df
    .filter(F.col("analysis_year") == 2019)
    .orderBy("poverty_rank")
)

In [0]:
province_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_province_comparison_v1"
)

if spark.catalog.tableExists(province_dashboard_target):
    print(
        f"Table already exists. Nothing was written: "
        f"{province_dashboard_target}"
    )
else:
    (
        cbsl_dashboard_province_comparison_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(province_dashboard_target)
    )

    print(
        f"Table created successfully: "
        f"{province_dashboard_target}"
    )

In [0]:
province_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_province_comparison_v1"
)

saved_province_dashboard_df = spark.table(province_dashboard_target)

print(f"Table: {province_dashboard_target}")
print(f"Exists: {spark.catalog.tableExists(province_dashboard_target)}")
print(f"Rows: {saved_province_dashboard_df.count()}")
print(f"Columns: {len(saved_province_dashboard_df.columns)}")

In [0]:
print("Provincial socioeconomic Gold table columns:\n")

for column_name in cbsl_province_socioeconomic_df.columns:
    print(column_name)

In [0]:
from pyspark.sql import functions as F

province_poverty_change_df = (
    spark.table(
        "workspace.census360.gold_cbsl_dashboard_province_comparison_v1"
    )
    .filter(F.col("analysis_year") == 2019)
    .select(
        "province_name",
        "poverty_headcount_change_pp"
    )
)

cbsl_dashboard_province_change_df = (
    cbsl_province_socioeconomic_df
    .filter(F.col("analysis_year") == 2019)
    .select(
        F.col("geography_name").alias("province_name"),
        F.col("previous_analysis_year"),
        F.col("analysis_year"),
        F.col("years_since_previous_observation"),

        F.col("mean_household_income_change_percent"),
        F.col("mean_per_person_income_change_percent"),
        F.col("median_household_income_change_percent"),
        F.col("household_expenditure_change_percent"),

        F.col("communication_expenditure_share_change_pp"),
        F.col("education_expenditure_share_change_pp"),
        F.col("transport_expenditure_share_change_pp"),
        F.col("housing_expenditure_share_change_pp"),
        F.col("household_gini_change_points")
    )
    .join(
        province_poverty_change_df,
        on="province_name",
        how="left"
    )
    .withColumn(
        "poverty_movement",
        F.when(
            F.col("poverty_headcount_change_pp") < 0,
            F.lit("Poverty decreased")
        ).when(
            F.col("poverty_headcount_change_pp") > 0,
            F.lit("Poverty increased")
        ).otherwise(
            F.lit("No poverty change")
        )
    )
    .withColumn(
        "change_segment",
        F.when(
            (F.col("mean_per_person_income_change_percent") > 0)
            & (F.col("poverty_headcount_change_pp") < 0),
            F.lit("Income increased / Poverty decreased")
        ).when(
            (F.col("mean_per_person_income_change_percent") > 0)
            & (F.col("poverty_headcount_change_pp") >= 0),
            F.lit("Income increased / Poverty did not decrease")
        ).otherwise(
            F.lit("Mixed change pattern")
        )
    )
    .orderBy(
        F.col("poverty_headcount_change_pp").desc()
    )
)

print(
    f"Province change dataset created: "
    f"{cbsl_dashboard_province_change_df.count()} rows, "
    f"{len(cbsl_dashboard_province_change_df.columns)} columns"
)

display(cbsl_dashboard_province_change_df)

In [0]:
from pyspark.sql import functions as F

province_change_validation = (
    cbsl_dashboard_province_change_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("province_name").alias("distinct_provinces"),
        F.min("previous_analysis_year").alias("start_year"),
        F.max("analysis_year").alias("end_year"),

        F.sum(
            F.when(
                F.col("poverty_headcount_change_pp").isNull(), 1
            ).otherwise(0)
        ).alias("null_poverty_change_values"),

        F.sum(
            F.when(
                F.col("mean_per_person_income_change_percent").isNull(), 1
            ).otherwise(0)
        ).alias("null_income_change_values"),

        F.sum(
            F.when(
                F.col("poverty_headcount_change_pp") < 0, 1
            ).otherwise(0)
        ).alias("provinces_with_poverty_decrease"),

        F.sum(
            F.when(
                F.col("poverty_headcount_change_pp") > 0, 1
            ).otherwise(0)
        ).alias("provinces_with_poverty_increase")
    )
)

duplicate_provinces = (
    cbsl_dashboard_province_change_df
    .groupBy("province_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(province_change_validation)
print(f"Duplicate province groups: {duplicate_provinces}")

In [0]:
province_change_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_province_change_v1"
)

if spark.catalog.tableExists(province_change_dashboard_target):
    print(f"Table already exists: {province_change_dashboard_target}")
else:
    print(f"Table name is available: {province_change_dashboard_target}")

In [0]:
print("Columns:")
for column_name in cbsl_dashboard_province_change_df.columns:
    print(column_name)

print("\nProvincial changes from 2016 to 2019:")
display(
    cbsl_dashboard_province_change_df
    .orderBy(
        F.col("poverty_headcount_change_pp").desc()
    )
)

In [0]:
province_change_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_province_change_v1"
)

if spark.catalog.tableExists(province_change_dashboard_target):
    print(
        f"Table already exists. Nothing was written: "
        f"{province_change_dashboard_target}"
    )
else:
    (
        cbsl_dashboard_province_change_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(province_change_dashboard_target)
    )

    print(
        f"Table created successfully: "
        f"{province_change_dashboard_target}"
    )

In [0]:
province_change_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_province_change_v1"
)

saved_province_change_df = spark.table(
    province_change_dashboard_target
)

print(f"Table: {province_change_dashboard_target}")
print(
    f"Exists: "
    f"{spark.catalog.tableExists(province_change_dashboard_target)}"
)
print(f"Rows: {saved_province_change_df.count()}")
print(f"Columns: {len(saved_province_change_df.columns)}")

In [0]:
from pyspark.sql import functions as F

display(
    cbsl_province_association_df
    .select(
        "association_type",
        "analysis_group",
        "indicator",
        "correlation"
    )
    .orderBy(
        "association_type",
        "analysis_group",
        F.col("correlation").asc()
    )
)

In [0]:
from pyspark.sql import functions as F

cbsl_dashboard_association_summary_df = (
    cbsl_province_association_df
    .withColumn(
        "association_indicator",
        F.when(
            F.col("indicator") == "mean_income_per_person_monthly_lkr",
            "Mean income per person"
        ).when(
            F.col("indicator") == "mean_income_per_household_monthly_lkr",
            "Mean household income"
        ).when(
            F.col("indicator") == "median_income_per_household_monthly_lkr",
            "Median household income"
        ).when(
            F.col("indicator") == "household_expenditure_monthly_lkr",
            "Household expenditure"
        ).when(
            F.col("indicator") == "expenditure_share_communication_percent",
            "Communication expenditure share"
        ).when(
            F.col("indicator") == "education_passed_gce_ol_percent",
            "GCE O/L attainment"
        ).when(
            F.col("indicator") == "education_passed_gce_al_and_above_percent",
            "GCE A/L and above attainment"
        ).when(
            F.col("indicator") == "gini_coefficient_households",
            "Household Gini"
        ).when(
            F.col("indicator") == "mean_household_income_change_percent",
            "Mean household income change"
        ).when(
            F.col("indicator") == "mean_per_person_income_change_percent",
            "Mean income per person change"
        ).when(
            F.col("indicator") == "median_household_income_change_percent",
            "Median household income change"
        ).when(
            F.col("indicator") == "household_expenditure_change_percent",
            "Household expenditure change"
        ).when(
            F.col("indicator") == "communication_expenditure_share_change_pp",
            "Communication expenditure share change"
        ).when(
            F.col("indicator") == "education_expenditure_share_change_pp",
            "Education expenditure share change"
        ).when(
            F.col("indicator") == "household_gini_change_points",
            "Household Gini change"
        ).otherwise(F.col("indicator"))
    )
    .withColumnRenamed("analysis_group", "association_period")
    .withColumnRenamed("correlation", "pearson_correlation")
    .withColumn(
        "absolute_correlation",
        F.round(F.abs(F.col("pearson_correlation")), 4)
    )
    .withColumn(
        "association_direction",
        F.when(
            F.col("pearson_correlation") < 0,
            "Negative"
        ).when(
            F.col("pearson_correlation") > 0,
            "Positive"
        ).otherwise("No observed association")
    )
    .withColumn(
        "association_strength_label",
        F.when(
            F.abs(F.col("pearson_correlation")) >= 0.70,
            "Strong"
        ).when(
            F.abs(F.col("pearson_correlation")) >= 0.40,
            "Moderate"
        ).otherwise("Weak")
    )
    .withColumn(
        "sample_size",
        F.when(
            F.col("association_period") == "All observations",
            F.lit(18)
        ).otherwise(F.lit(9))
    )
    .withColumn(
        "target_measure",
        F.when(
            F.col("association_type") == "change",
            "Poverty headcount change"
        ).otherwise("Poverty headcount level")
    )
    .withColumn(
        "interpretation_note",
        F.concat(
            F.lit("Exploratory "),
            F.lower(F.col("association_direction")),
            F.lit(" association with "),
            F.lower(F.col("target_measure")),
            F.lit("; sample size = "),
            F.col("sample_size"),
            F.lit(". This does not establish causation.")
        )
    )
    .select(
        "association_type",
        "association_period",
        "association_indicator",
        "target_measure",
        "pearson_correlation",
        "absolute_correlation",
        "association_direction",
        "association_strength_label",
        "sample_size",
        "interpretation_note"
    )
    .orderBy(
        "association_type",
        "association_period",
        F.col("absolute_correlation").desc()
    )
)

print(
    f"Association dashboard dataset created: "
    f"{cbsl_dashboard_association_summary_df.count()} rows, "
    f"{len(cbsl_dashboard_association_summary_df.columns)} columns"
)

display(cbsl_dashboard_association_summary_df)

In [0]:
from pyspark.sql import functions as F

association_validation = (
    cbsl_dashboard_association_summary_df
    .agg(
        F.count("*").alias("row_count"),

        F.sum(
            F.when(F.col("pearson_correlation").isNull(), 1).otherwise(0)
        ).alias("null_correlations"),

        F.sum(
            F.when(
                (F.col("pearson_correlation") < -1)
                | (F.col("pearson_correlation") > 1),
                1
            ).otherwise(0)
        ).alias("invalid_correlation_values"),

        F.sum(
            F.when(F.col("association_indicator").isNull(), 1).otherwise(0)
        ).alias("null_indicator_labels"),

        F.sum(
            F.when(F.col("interpretation_note").isNull(), 1).otherwise(0)
        ).alias("null_interpretation_notes")
    )
)

duplicate_associations = (
    cbsl_dashboard_association_summary_df
    .groupBy(
        "association_type",
        "association_period",
        "association_indicator"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

sample_size_validation = (
    cbsl_dashboard_association_summary_df
    .groupBy(
        "association_type",
        "association_period",
        "sample_size"
    )
    .count()
    .orderBy(
        "association_type",
        "association_period"
    )
)

display(association_validation)
print(f"Duplicate association groups: {duplicate_associations}")
display(sample_size_validation)

In [0]:
duplicate_associations = (
    cbsl_dashboard_association_summary_df
    .groupBy(
        "association_type",
        "association_period",
        "association_indicator"
    )
    .count()
    .filter("count > 1")
    .count()
)

print(f"Duplicate association groups: {duplicate_associations}")

In [0]:
association_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_association_summary_v1"
)

if spark.catalog.tableExists(association_dashboard_target):
    print(f"Table already exists: {association_dashboard_target}")
else:
    print(f"Table name is available: {association_dashboard_target}")

In [0]:
from pyspark.sql import functions as F

print("Columns:")
for column_name in cbsl_dashboard_association_summary_df.columns:
    print(column_name)

print("\nStrongest associations:")
display(
    cbsl_dashboard_association_summary_df
    .orderBy(F.col("absolute_correlation").desc())
    .limit(15)
)

In [0]:
association_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_association_summary_v1"
)

if spark.catalog.tableExists(association_dashboard_target):
    print(
        f"Table already exists. Nothing was written: "
        f"{association_dashboard_target}"
    )
else:
    (
        cbsl_dashboard_association_summary_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(association_dashboard_target)
    )

    print(
        f"Table created successfully: "
        f"{association_dashboard_target}"
    )

In [0]:
association_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_association_summary_v1"
)

saved_association_dashboard_df = spark.table(
    association_dashboard_target
)

print(f"Table: {association_dashboard_target}")
print(
    f"Exists: "
    f"{spark.catalog.tableExists(association_dashboard_target)}"
)
print(f"Rows: {saved_association_dashboard_df.count()}")
print(f"Columns: {len(saved_association_dashboard_df.columns)}")

In [0]:
from pyspark.sql import functions as F

telecom_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_telecom_trend_v1"
)

hies_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_hies_trend_v1"
)

poverty_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_national_poverty_v1"
)

province_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_province_comparison_v1"
)

association_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_association_summary_v1"
)

latest_telecom = telecom_df.orderBy(
    F.col("analysis_year").desc()
).first()

latest_hies = hies_df.orderBy(
    F.col("analysis_year").desc()
).first()

latest_comparable_poverty = (
    poverty_df
    .filter(F.col("safe_for_historical_change_analysis") == True)
    .orderBy(F.col("analysis_year").desc())
    .first()
)

updated_definition_poverty = (
    poverty_df
    .filter(F.col("requires_definition_warning") == True)
    .orderBy(F.col("analysis_year").desc())
    .first()
)

latest_province_year = (
    province_df
    .agg(F.max("analysis_year").alias("latest_year"))
    .first()["latest_year"]
)

highest_poverty_province = (
    province_df
    .filter(F.col("analysis_year") == latest_province_year)
    .orderBy(F.col("poverty_headcount_percent").desc())
    .first()
)

lowest_poverty_province = (
    province_df
    .filter(F.col("analysis_year") == latest_province_year)
    .orderBy(F.col("poverty_headcount_percent").asc())
    .first()
)

strongest_overall_association = (
    association_df
    .filter(
        (F.col("association_type") == "level")
        & (F.col("association_period") == "All observations")
    )
    .orderBy(F.col("absolute_correlation").desc())
    .first()
)

kpi_rows = [
    (
        "Latest telecom year",
        str(latest_telecom["analysis_year"]),
        "Year",
        latest_telecom["dashboard_status_label"],
        "Latest available CBSL telecom observation."
    ),
    (
        "Latest internet connections",
        f"{latest_telecom['internet_connections_including_mobile_total']:,.0f}",
        "Connections",
        str(latest_telecom["analysis_year"]),
        "Includes mobile internet connections."
    ),
    (
        "Latest cellular penetration",
        f"{latest_telecom['cellular_penetration_per_100']:,.1f}",
        "Per 100 people",
        str(latest_telecom["analysis_year"]),
        "Subscription penetration indicator."
    ),
    (
        "Latest fixed-access subscribers",
        f"{latest_telecom['fixed_access_subscribers_thousand']:,.0f}",
        "Thousand subscribers",
        str(latest_telecom["analysis_year"]),
        "Fixed-access subscriber indicator; not automatically fixed broadband."
    ),
    (
        "Latest mean household income",
        f"{latest_hies['mean_household_income_lkr']:,.0f}",
        "LKR per month",
        latest_hies["survey_period"],
        "Nominal value; not adjusted for inflation."
    ),
    (
        "Latest mean household expenditure",
        f"{latest_hies['mean_household_expenditure_lkr']:,.0f}",
        "LKR per month",
        latest_hies["survey_period"],
        "Nominal value from an irregular HIES survey observation."
    ),
    (
        "Latest comparable national poverty",
        f"{latest_comparable_poverty['poverty_headcount_percent']:,.1f}",
        "Percent",
        latest_comparable_poverty["dashboard_observation_label"],
        "Comparable with the older poverty-line historical series."
    ),
    (
        "Updated-line national poverty",
        f"{updated_definition_poverty['poverty_headcount_percent']:,.1f}",
        "Percent",
        updated_definition_poverty["dashboard_observation_label"],
        "Uses a different poverty-line definition and must remain separate."
    ),
    (
        "Highest-poverty province",
        highest_poverty_province["province_name"],
        f"{highest_poverty_province['poverty_headcount_percent']:,.1f}%",
        str(latest_province_year),
        "Highest observed provincial poverty headcount in the comparable year."
    ),
    (
        "Lowest-poverty province",
        lowest_poverty_province["province_name"],
        f"{lowest_poverty_province['poverty_headcount_percent']:,.1f}%",
        str(latest_province_year),
        "Lowest observed provincial poverty headcount in the comparable year."
    ),
    (
        "Strongest overall association",
        strongest_overall_association["association_indicator"],
        f"{strongest_overall_association['pearson_correlation']:+.4f}",
        f"n={strongest_overall_association['sample_size']}",
        "Exploratory association only; it does not establish causation."
    ),
    (
        "Comparable province-year observations",
        str(province_df.count()),
        "Observations",
        "2016 and 2019",
        "Nine provinces observed in two years."
    )
]

cbsl_dashboard_kpi_summary_df = spark.createDataFrame(
    kpi_rows,
    [
        "kpi_name",
        "kpi_value",
        "unit_or_secondary_value",
        "observation_context",
        "interpretation_note"
    ]
)

print(
    f"KPI summary created: "
    f"{cbsl_dashboard_kpi_summary_df.count()} rows, "
    f"{len(cbsl_dashboard_kpi_summary_df.columns)} columns"
)

display(cbsl_dashboard_kpi_summary_df)

In [0]:
from pyspark.sql import functions as F

kpi_validation = (
    cbsl_dashboard_kpi_summary_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("kpi_name").alias("distinct_kpi_names"),

        F.sum(
            F.when(F.col("kpi_name").isNull(), 1).otherwise(0)
        ).alias("null_kpi_names"),

        F.sum(
            F.when(F.col("kpi_value").isNull(), 1).otherwise(0)
        ).alias("null_kpi_values"),

        F.sum(
            F.when(F.col("observation_context").isNull(), 1).otherwise(0)
        ).alias("null_observation_contexts"),

        F.sum(
            F.when(F.col("interpretation_note").isNull(), 1).otherwise(0)
        ).alias("null_interpretation_notes")
    )
)

duplicate_kpis = (
    cbsl_dashboard_kpi_summary_df
    .groupBy("kpi_name")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(kpi_validation)
print(f"Duplicate KPI names: {duplicate_kpis}")

In [0]:
duplicate_kpis = (
    cbsl_dashboard_kpi_summary_df
    .groupBy("kpi_name")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Duplicate KPI names: {duplicate_kpis}")

In [0]:
kpi_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_kpi_summary_v1"
)

if spark.catalog.tableExists(kpi_dashboard_target):
    print(f"Table already exists: {kpi_dashboard_target}")
else:
    print(f"Table name is available: {kpi_dashboard_target}")

In [0]:
print("Columns:")
for column_name in cbsl_dashboard_kpi_summary_df.columns:
    print(column_name)

print("\nExecutive KPI summary:")
display(
    cbsl_dashboard_kpi_summary_df
    .orderBy("kpi_name")
)

In [0]:
kpi_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_kpi_summary_v1"
)

if spark.catalog.tableExists(kpi_dashboard_target):
    print(
        f"Table already exists. Nothing was written: "
        f"{kpi_dashboard_target}"
    )
else:
    (
        cbsl_dashboard_kpi_summary_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(kpi_dashboard_target)
    )

    print(
        f"Table created successfully: "
        f"{kpi_dashboard_target}"
    )

In [0]:
kpi_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_kpi_summary_v1"
)

saved_kpi_dashboard_df = spark.table(kpi_dashboard_target)

print(f"Table: {kpi_dashboard_target}")
print(f"Exists: {spark.catalog.tableExists(kpi_dashboard_target)}")
print(f"Rows: {saved_kpi_dashboard_df.count()}")
print(f"Columns: {len(saved_kpi_dashboard_df.columns)}")

In [0]:
from pyspark.sql import functions as F

telecom_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_telecom_trend_v1"
)

hies_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_hies_trend_v1"
)

poverty_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_national_poverty_v1"
)

province_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_province_comparison_v1"
)

province_change_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_province_change_v1"
)

association_df = spark.table(
    "workspace.census360.gold_cbsl_dashboard_association_summary_v1"
)

first_telecom = telecom_df.orderBy("analysis_year").first()
latest_telecom = telecom_df.orderBy(F.col("analysis_year").desc()).first()

first_hies = hies_df.orderBy("analysis_year").first()
latest_hies = hies_df.orderBy(F.col("analysis_year").desc()).first()

internet_growth = (
    (
        latest_telecom["internet_connections_including_mobile_total"]
        / first_telecom["internet_connections_including_mobile_total"]
    ) - 1
) * 100

fixed_access_growth = (
    (
        latest_telecom["fixed_access_subscribers_thousand"]
        / first_telecom["fixed_access_subscribers_thousand"]
    ) - 1
) * 100

nominal_income_growth = (
    (
        latest_hies["mean_household_income_lkr"]
        / first_hies["mean_household_income_lkr"]
    ) - 1
) * 100

comparable_poverty = (
    poverty_df
    .filter(F.col("safe_for_historical_change_analysis") == True)
    .orderBy("analysis_year")
    .collect()
)

poverty_decline_pp = (
    comparable_poverty[-1]["poverty_headcount_percent"]
    - comparable_poverty[0]["poverty_headcount_percent"]
)

latest_province_year = province_df.agg(
    F.max("analysis_year").alias("year")
).first()["year"]

highest_poverty = (
    province_df
    .filter(F.col("analysis_year") == latest_province_year)
    .orderBy(F.col("poverty_headcount_percent").desc())
    .first()
)

largest_rank_gap = (
    province_df
    .filter(F.col("analysis_year") == latest_province_year)
    .orderBy(F.col("income_poverty_rank_gap").desc())
    .first()
)

poverty_decrease_count = (
    province_change_df
    .filter(F.col("poverty_headcount_change_pp") < 0)
    .count()
)

poverty_increase_provinces = [
    row["province_name"]
    for row in (
        province_change_df
        .filter(F.col("poverty_headcount_change_pp") > 0)
        .select("province_name")
        .orderBy("province_name")
        .collect()
    )
]

strongest_association = (
    association_df
    .filter(
        (F.col("association_type") == "level")
        & (F.col("association_period") == "All observations")
    )
    .orderBy(F.col("absolute_correlation").desc())
    .first()
)

client_insight_rows = [
    (
        1,
        "Digital connectivity expanded strongly",
        f"Internet connections increased by {internet_growth:,.1f}% "
        f"between {first_telecom['analysis_year']} and "
        f"{latest_telecom['analysis_year']}.",
        "Telecom",
        "Includes mobile internet connections; the latest observation is provisional."
    ),
    (
        2,
        "Fixed-access subscribers moved differently",
        f"The fixed-access subscriber indicator changed by "
        f"{fixed_access_growth:+,.1f}% over the telecom period.",
        "Telecom",
        "Do not describe this indicator automatically as fixed broadband."
    ),
    (
        3,
        "Household income rose substantially in nominal terms",
        f"Mean household income increased by {nominal_income_growth:,.1f}% "
        f"between {first_hies['survey_period']} and {latest_hies['survey_period']}.",
        "Households",
        "The values are nominal and are not adjusted for inflation."
    ),
    (
        4,
        "Comparable national poverty declined",
        f"The comparable poverty headcount moved from "
        f"{comparable_poverty[0]['poverty_headcount_percent']:.1f}% to "
        f"{comparable_poverty[-1]['poverty_headcount_percent']:.1f}% "
        f"({poverty_decline_pp:+.1f} percentage points).",
        "Poverty",
        "Uses only the older comparable poverty-line series."
    ),
    (
        5,
        "The updated poverty definition changes interpretation",
        "The 2019(b) poverty observation must remain separate from the "
        "2012/13, 2016 and 2019(a) historical series.",
        "Poverty",
        "The apparent difference reflects a different poverty-line definition."
    ),
    (
        6,
        "Northern Province had the highest comparable poverty",
        f"{highest_poverty['province_name']} recorded "
        f"{highest_poverty['poverty_headcount_percent']:.1f}% poverty in "
        f"{latest_province_year}.",
        "Provincial",
        "This is a descriptive provincial comparison."
    ),
    (
        7,
        "Most provinces recorded poverty reduction",
        f"Poverty declined in {poverty_decrease_count} of 9 provinces "
        f"between 2016 and 2019.",
        "Provincial change",
        "Provincial changes use the comparable poverty definition."
    ),
    (
        8,
        "Income growth did not guarantee poverty reduction",
        f"Poverty increased slightly in "
        f"{', '.join(poverty_increase_provinces)} despite income growth.",
        "Provincial change",
        "This is an observed pattern and does not establish causation."
    ),
    (
        9,
        "Income per person had the strongest observed association",
        f"{strongest_association['association_indicator']} had a correlation "
        f"of {strongest_association['pearson_correlation']:+.4f} with poverty.",
        "Association",
        f"Exploratory result based on n={strongest_association['sample_size']} observations."
    ),
    (
        10,
        "Eastern Province showed the largest rank mismatch",
        f"{largest_rank_gap['province_name']} had an income–poverty rank gap "
        f"of {largest_rank_gap['income_poverty_rank_gap']}.",
        "Provincial",
        "Useful for client drill-down and further contextual investigation."
    )
]

cbsl_dashboard_client_insights_df = spark.createDataFrame(
    client_insight_rows,
    [
        "insight_order",
        "insight_title",
        "insight_summary",
        "insight_category",
        "interpretation_note"
    ]
)

print(
    f"Client insight summary created: "
    f"{cbsl_dashboard_client_insights_df.count()} rows, "
    f"{len(cbsl_dashboard_client_insights_df.columns)} columns"
)

display(
    cbsl_dashboard_client_insights_df
    .orderBy("insight_order")
)

In [0]:
from pyspark.sql import functions as F

client_insight_validation = (
    cbsl_dashboard_client_insights_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("insight_order").alias("distinct_insight_orders"),
        F.countDistinct("insight_title").alias("distinct_insight_titles"),

        F.sum(
            F.when(F.col("insight_title").isNull(), 1).otherwise(0)
        ).alias("null_insight_titles"),

        F.sum(
            F.when(F.col("insight_summary").isNull(), 1).otherwise(0)
        ).alias("null_insight_summaries"),

        F.sum(
            F.when(F.col("interpretation_note").isNull(), 1).otherwise(0)
        ).alias("null_interpretation_notes")
    )
)

duplicate_insights = (
    cbsl_dashboard_client_insights_df
    .groupBy("insight_order", "insight_title")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(client_insight_validation)
print(f"Duplicate insight records: {duplicate_insights}")

In [0]:
duplicate_insights = (
    cbsl_dashboard_client_insights_df
    .groupBy("insight_order", "insight_title")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Duplicate insight records: {duplicate_insights}")

In [0]:
client_insights_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_client_insights_v1"
)

if spark.catalog.tableExists(client_insights_dashboard_target):
    print(f"Table already exists: {client_insights_dashboard_target}")
else:
    print(f"Table name is available: {client_insights_dashboard_target}")

In [0]:
print("Columns:")
for column_name in cbsl_dashboard_client_insights_df.columns:
    print(column_name)

print("\nClient insights:")
display(
    cbsl_dashboard_client_insights_df
    .orderBy("insight_order")
)

In [0]:
client_insights_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_client_insights_v1"
)

if spark.catalog.tableExists(client_insights_dashboard_target):
    print(
        f"Table already exists. Nothing was written: "
        f"{client_insights_dashboard_target}"
    )
else:
    (
        cbsl_dashboard_client_insights_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(client_insights_dashboard_target)
    )

    print(
        f"Table created successfully: "
        f"{client_insights_dashboard_target}"
    )

In [0]:
client_insights_dashboard_target = (
    "workspace.census360.gold_cbsl_dashboard_client_insights_v1"
)

saved_client_insights_df = spark.table(
    client_insights_dashboard_target
)

print(f"Table: {client_insights_dashboard_target}")
print(
    f"Exists: "
    f"{spark.catalog.tableExists(client_insights_dashboard_target)}"
)
print(f"Rows: {saved_client_insights_df.count()}")
print(f"Columns: {len(saved_client_insights_df.columns)}")

In [0]:
dashboard_note_rows = [
    (
        1,
        "Telecom",
        "Internet connections",
        "Internet connections include mobile internet connections. Do not label this metric as fixed or home broadband."
    ),
    (
        2,
        "Telecom",
        "Fixed-access subscribers",
        "The fixed subscriber base is a fixed-access indicator and must not automatically be described as fixed broadband."
    ),
    (
        3,
        "Telecom",
        "Data status",
        "The 2023 telecom observation is revised and the 2024 observation is provisional."
    ),
    (
        4,
        "HIES",
        "Irregular survey periods",
        "HIES observations come from irregular survey periods. Missing years were not interpolated and annual values were not invented."
    ),
    (
        5,
        "HIES",
        "Nominal monetary values",
        "Income and expenditure values are nominal and are not adjusted for inflation."
    ),
    (
        6,
        "National poverty",
        "Comparable historical series",
        "Use 2012/13, 2016 and 2019(a) for the older comparable poverty-line series."
    ),
    (
        7,
        "National poverty",
        "2019 definition change",
        "The 2019(b) observation uses an updated poverty-line definition and must remain separate from the older series."
    ),
    (
        8,
        "Provincial analysis",
        "Geographic coverage",
        "The integrated socioeconomic comparison contains nine provinces for 2016 and 2019. District drill-down is not available."
    ),
    (
        9,
        "Associations",
        "Exploratory correlations",
        "Correlation results describe observed associations only and do not establish causation."
    ),
    (
        10,
        "Associations",
        "Small sample size",
        "Overall provincial correlations use 18 observations, while individual-year and change analyses use only nine observations."
    ),
    (
        11,
        "Modelling",
        "Regression and machine learning",
        "Regression and machine-learning models were not used because the overlapping datasets and provincial samples were too small."
    ),
    (
        12,
        "Dashboard",
        "Client interpretation",
        "Dashboard insights should support investigation and decision-making, but they should not be treated as statistical proof or predictions."
    )
]

cbsl_dashboard_notes_df = spark.createDataFrame(
    dashboard_note_rows,
    [
        "note_order",
        "note_category",
        "note_topic",
        "note_text"
    ]
)

print(
    f"Dashboard notes dataset created: "
    f"{cbsl_dashboard_notes_df.count()} rows, "
    f"{len(cbsl_dashboard_notes_df.columns)} columns"
)

display(
    cbsl_dashboard_notes_df
    .orderBy("note_order")
)

In [0]:
from pyspark.sql import functions as F

dashboard_notes_validation = (
    cbsl_dashboard_notes_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("note_order").alias("distinct_note_orders"),

        F.sum(
            F.when(F.col("note_category").isNull(), 1).otherwise(0)
        ).alias("null_note_categories"),

        F.sum(
            F.when(F.col("note_topic").isNull(), 1).otherwise(0)
        ).alias("null_note_topics"),

        F.sum(
            F.when(F.col("note_text").isNull(), 1).otherwise(0)
        ).alias("null_note_texts")
    )
)

duplicate_notes = (
    cbsl_dashboard_notes_df
    .groupBy("note_order", "note_topic")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(dashboard_notes_validation)
print(f"Duplicate dashboard notes: {duplicate_notes}")

In [0]:
duplicate_notes = (
    cbsl_dashboard_notes_df
    .groupBy("note_order", "note_topic")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Duplicate dashboard notes: {duplicate_notes}")

In [0]:
dashboard_notes_target = (
    "workspace.census360.gold_cbsl_dashboard_notes_v1"
)

if spark.catalog.tableExists(dashboard_notes_target):
    print(f"Table already exists: {dashboard_notes_target}")
else:
    print(f"Table name is available: {dashboard_notes_target}")

In [0]:
print("Columns:")
for column_name in cbsl_dashboard_notes_df.columns:
    print(column_name)

print("\nDashboard notes and limitations:")
display(
    cbsl_dashboard_notes_df
    .orderBy("note_order")
)

In [0]:
dashboard_notes_target = (
    "workspace.census360.gold_cbsl_dashboard_notes_v1"
)

if spark.catalog.tableExists(dashboard_notes_target):
    print(
        f"Table already exists. Nothing was written: "
        f"{dashboard_notes_target}"
    )
else:
    (
        cbsl_dashboard_notes_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(dashboard_notes_target)
    )

    print(
        f"Table created successfully: "
        f"{dashboard_notes_target}"
    )

In [0]:
dashboard_notes_target = (
    "workspace.census360.gold_cbsl_dashboard_notes_v1"
)

saved_dashboard_notes_df = spark.table(
    dashboard_notes_target
)

print(f"Table: {dashboard_notes_target}")
print(f"Exists: {spark.catalog.tableExists(dashboard_notes_target)}")
print(f"Rows: {saved_dashboard_notes_df.count()}")
print(f"Columns: {len(saved_dashboard_notes_df.columns)}")

In [0]:
dashboard_tables = {
    "Telecom Trend":
        "workspace.census360.gold_cbsl_dashboard_telecom_trend_v1",

    "HIES Trend":
        "workspace.census360.gold_cbsl_dashboard_hies_trend_v1",

    "National Poverty":
        "workspace.census360.gold_cbsl_dashboard_national_poverty_v1",

    "Province Comparison":
        "workspace.census360.gold_cbsl_dashboard_province_comparison_v1",

    "Province Change":
        "workspace.census360.gold_cbsl_dashboard_province_change_v1",

    "Association Summary":
        "workspace.census360.gold_cbsl_dashboard_association_summary_v1",

    "KPI Summary":
        "workspace.census360.gold_cbsl_dashboard_kpi_summary_v1",

    "Client Insights":
        "workspace.census360.gold_cbsl_dashboard_client_insights_v1",

    "Dashboard Notes":
        "workspace.census360.gold_cbsl_dashboard_notes_v1"
}

print("Final CBSL dashboard-table verification:\n")

for table_label, table_name in dashboard_tables.items():
    exists = spark.catalog.tableExists(table_name)

    if exists:
        table_df = spark.table(table_name)

        print(
            f"{table_label}: "
            f"Exists=True | "
            f"Rows={table_df.count()} | "
            f"Columns={len(table_df.columns)}"
        )
    else:
        print(f"{table_label}: Exists=False")